# Entrenamiento del modelo de reconocimiento de robots

In [1]:
!pip install ultralytics -q
from ultralytics import YOLO
import ultralytics
ultralytics.checks()

Ultralytics 8.4.63 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 47.1/112.6 GB disk)


In [1]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="e7KptQaiu8Ewv1Wng1yS")
project = rf.workspace("supervisoin").project("vision-computacional-ghs1n")
dataset = project.version(2).download("yolov8")

loading Roboflow workspace...
loading Roboflow project...


# Crear el split manualmente

In [11]:
import os, shutil, random

img_src = "Vision-Computacional-2/train/images"
lbl_src = "Vision-Computacional-2/train/labels"

for split in ["train", "valid", "test"]:
    os.makedirs(f"dataset/{split}/images", exist_ok=True)
    os.makedirs(f"dataset/{split}/labels", exist_ok=True)

imagenes = [f for f in os.listdir(img_src) if f.endswith((".jpg", ".png"))]
random.seed(42)  # para reproducibilidad
random.shuffle(imagenes)

n = len(imagenes)
train_imgs = imagenes[:int(n * 0.70)]
valid_imgs = imagenes[int(n * 0.70):int(n * 0.90)]
test_imgs  = imagenes[int(n * 0.90):]

for split, imgs in [("train", train_imgs), ("valid", valid_imgs), ("test", test_imgs)]:
    for img in imgs:
        shutil.copy(f"{img_src}/{img}", f"dataset/{split}/images/{img}")
        label = os.path.splitext(img)[0] + ".txt"
        if os.path.exists(f"{lbl_src}/{label}"):
            shutil.copy(f"{lbl_src}/{label}", f"dataset/{split}/labels/{label}")

print(f"✓ Train: {len(train_imgs)}")
print(f"✓ Valid: {len(valid_imgs)}")
print(f"✓ Test:  {len(test_imgs)}")

✓ Train: 2161
✓ Valid: 618
✓ Test:  309


In [12]:
yaml_content = """names:
- ball
- robot
nc: 2
train: dataset/train/images
val: dataset/valid/images
test: dataset/test/images
"""

with open("dataset/data.yaml", "w") as f:
    f.write(yaml_content)

print("✓ data.yaml listo")

✓ data.yaml listo


In [14]:
import os

# Ruta absoluta en Colab
base = "/content"

yaml_content = f"""names:
- ball
- robot
nc: 2
train: {base}/dataset/train/images
val: {base}/dataset/valid/images
test: {base}/dataset/test/images
"""

with open("dataset/data.yaml", "w") as f:
    f.write(yaml_content)

# Verificar que las carpetas existen
for split in ["train", "valid", "test"]:
    ruta = f"{base}/dataset/{split}/images"
    n = len(os.listdir(ruta)) if os.path.exists(ruta) else 0
    print(f"{'✓' if n > 0 else '✗'} {ruta} — {n} imágenes")

✓ /content/dataset/train/images — 2812 imágenes
✓ /content/dataset/valid/images — 1116 imágenes
✓ /content/dataset/test/images — 585 imágenes


In [15]:
from ultralytics import YOLO

model = YOLO("yolov8m.pt")

results = model.train(
    data="dataset/data.yaml",
    epochs=50,
    imgsz=640,
    batch=8,
    patience=10,
    device=0,
    project="runs/train",
    name="robots_v1",
    exist_ok=True
)

print(f"\nmAP50: {results.results_dict['metrics/mAP50(B)']:.3f}")
print(f"Modelo: runs/train/robots_v1/weights/best.pt")

Ultralytics 8.4.63 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=robots_v1, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=10, perspective=0

In [16]:
import shutil

shutil.copy(
    "/content/runs/detect/runs/train/robots_v1/weights/best.pt",
    "yucabot.pt"
)

print("✓ yucabot.pt listo")

✓ yucabot.pt listo


In [17]:
from google.colab import files
files.download("yucabot.pt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>